# Topical Lectures, Controls
## 2D simulator, part 4: adding a velocity loop

Andreas Freise, 30.10.2026

In the basic cascade the output of the y position loop was used directly as the
desired tilt angle. That output has no physical unit: it is the position
error in metres times a gain, and we limited it to $\pm 0.1\pi$ to keep
the tilt small. How far the drone tilts for a given distance to the
target is then set by the gain alone, and the speed the drone reaches
follows from the tilt, with no limit of its own.

In this notebook we insert a *velocity loop* between the position
controller and the tilt controller. The cascade becomes:

```
position PID  -> desired velocity (m/s, clipped at +-1 m/s)
velocity PID  -> desired tilt    (rad,   clipped at +-20 deg)
phi PID       -> motor differential (the torque)
```

The output of each loop now has a physical unit, and the saturation
limits have a direct meaning. This is the structure of the controller in
the Crazyflie firmware (Section 3 of the paper cited in
`drone_parameters.md`), apart from a fourth loop on the attitude rate at 500 Hz
inside the microcontroller, which we do not model here.


In [ ]:
# Importing packages
import numpy as np
import matplotlib.pyplot as plt
import time
import module
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

%matplotlib inline

MyName = "Caspar"


## Drone parameters and helpers

The same constants and motor-mixing helpers as before.


In [ ]:
# Calibration constants of your drone, from the system identification:
# copy the values printed at the end of that notebook here.
# The numbers below are those of the drone of "Caspar".
V_left_offset = 0.00135
V_hover = 0.5331
V_max = 1.0
V_min = 0

def set_v_nohover(drone, _V_left, _V_right):
    V_left = _V_left + V_left_offset
    V_right = _V_right
    fb_z = (V_left + V_right) / 2
    fb_phi = (V_left - V_right) / 2
    fb_z = max(V_min, min(fb_z, V_max))
    headroom = min(fb_z - V_min, V_max - fb_z)
    fb_phi = max(-headroom, min(fb_phi, headroom))
    drone.set_V(fb_z + fb_phi, fb_z - fb_phi)

def zphi2V(fb_z, fb_phi):
    return fb_z + fb_phi, fb_z - fb_phi


## The velocity-loop cascade

Why a loop inside a loop? Think of walking to a door. Your eyes judge the distance and decide how fast to walk: far away, normal pace; close, slow down. Your legs turn that pace into steps and lean, and you never think about the angle of your body. The position loop is the eyes: it looks at the distance and outputs a desired velocity. The velocity loop is the legs: it looks at the set velocity and outputs the tilt that speeds up or slows down. The tilt loop holds the body at that angle. Each loop focusses on one quantity and only has to make it follow the number it is given. That works because every loop is faster than the one outside it: the tilt settles much faster than the velocity, the velocity faster than the position. And the clip of the magnitudes after each loop now has a meaning in physical units: no faster than 1 m/s, no steeper than 20 degrees.

We use the same `module.PIDController` class four times: one for each
position axis, one for each velocity axis. The phi controller is the
same PD as in the basic cascade.

Notation: a subscript 0 (`y0`, `z0`, `phi0`) is a setpoint we give from outside, the target. A `_des` suffix (`dy_des`, `dz_des`, `phi_des`) is a setpoint computed by the loop outside it, the *desired* value that the next loop has to deliver.

Sign convention for the new loops is worth working out explicitly:

* **Position loop (y or z):** `err = state.y - y0`. If the drone is
  *above* its setpoint we want a *negative* desired velocity, so we
  pick a *negative* `Kp` for the position controller.
* **Velocity loop, y:** `err = state.dy - dy_des`. If the drone is
  moving right faster than its velocity setpoint, we want positive tilt (drone
  tips right, accelerates left). Positive `Kp`.
* **Velocity loop, z:** `err = state.dz - dz_des`. If the drone is
  climbing faster than its velocity setpoint, we want less thrust. Negative `Kp`.

The saturation limits are taken from the CF firmware: $\pm 1$ m/s
on the velocity setpoint and $\pm 20°$ on the tilt setpoint.


In [ ]:
# Position -> velocity (P-only is fine; the integrator lives in the
# inner velocity loop, where it can compensate steady-state errors).
y_pos_ctrl = module.PIDController(Kp=-1.6, Ki=0, Kd=0)
z_pos_ctrl = module.PIDController(Kp=-5.0, Ki=0, Kd=0)

# Velocity -> tilt / thrust correction. PI: integrator handles biases.
y_vel_ctrl = module.PIDController(Kp=1.2,  Ki=0.02, Kd=0)
z_vel_ctrl = module.PIDController(Kp=-0.5, Ki=-0.1, Kd=0)

# Inner phi PD, as in the basic cascade.
phi_ctrl   = module.PIDController(Kp=0.1, Ki=0, Kd=0.015)

# Saturation limits, taken from the CF firmware.
v_max    = 1.0                 # m/s, the limit on the velocity setpoint
tilt_max = 20 * np.pi / 180    # rad, the limit on the tilt setpoint (20 deg)


## A large position step with and without the velocity loop

To see the difference between the flat cascade without velocity loop and the velocity-loop cascade above, we step the y-setpoint from 0 to 2.5 m and run the simulation twice with the same drone, once with each controller. The z-setpoint stays at zero.

Two simulator settings matter for this demonstration:

* `wind=False`, so the comparison is about the controllers, not about disturbance rejection.
* `drag_t=0`, so that the speed is set by the controller alone and not limited by the drag in the simulator. The later notebooks keep the drag on, as it is part of a realistic model of the Crazyflie.

For the **flat cascade** (no velocity loop), the position PID's output is interpreted directly as a tilt setpoint and clipped at $\pm 20°$. The drone tilts to the limit and accelerates until the position error changes sign after the overshoot.

For the **velocity-loop cascade**, the position PID's output is interpreted as a desired *velocity*, clipped at $\pm v_{\mathrm{max}} = 1$ m/s. The inner velocity loop then turns that into the tilt needed to reach $v_{\mathrm{max}}$, and stops adding tilt once the drone is there. The drone moves at the speed limit, decelerates and settles at the setpoint.

In [ ]:
# Step from y=0 to y=2.5, hold z=0. Wind off and drag off so we see
# the controllers' behaviour rather than the simulator's drag.
y0 = 2.5
z0 = 0.0
T_run = 8.0
N = int(T_run * 60)


# --- Flat cascade (no velocity loop) ---------------------------
# The position PID sets the tilt setpoint directly, clipped at +-20 deg.
y_PID_flat   = module.PIDController(Kp=1.5,  Ki=0,    Kd=0.3)
z_PID_flat   = module.PIDController(Kp=-2.0, Ki=-0.2, Kd=-1.0)
phi_PID_flat = module.PIDController(Kp=0.1,  Ki=0,    Kd=0.015)
tilt_max_flat = np.deg2rad(20)

drone = module.Drone(name=MyName, wind=False,
                     flight_range=(-5, -3, 5, 3),
                     drag_t=0, drag_r=5)
results_flat = np.zeros([N, 14])
V_l = V_r = 0
for ctrl in (y_PID_flat, z_PID_flat, phi_PID_flat):
    ctrl.reset()
for i in range(N):
    state = drone.update()
    results_flat[i, 0:12] = module.state_to_array(state)
    results_flat[i, 12:14] = [V_l, V_r]

    # Position PID -> tilt setpoint directly.
    phi_des = float(np.clip(y_PID_flat(state.y, state.dy, y0),
                             -tilt_max_flat, tilt_max_flat))
    z_corr   = z_PID_flat(state.z, state.dz, z0)
    phi_fb   = phi_PID_flat(state.phi, state.dphi, phi_des)
    hover_ff = V_hover / np.cos(state.phi)
    V_l, V_r = zphi2V(hover_ff + z_corr, phi_fb)
    set_v_nohover(drone, V_l, V_r)


# --- Velocity-loop cascade (defined above) -------------------------
drone = module.Drone(name=MyName, wind=False,
                     flight_range=(-5, -3, 5, 3),
                     drag_t=0, drag_r=5)
results_vel = np.zeros([N, 14])
V_l = V_r = 0
for ctrl in (y_pos_ctrl, z_pos_ctrl, y_vel_ctrl, z_vel_ctrl, phi_ctrl):
    ctrl.reset()
for i in range(N):
    state = drone.update()
    results_vel[i, 0:12] = module.state_to_array(state)
    results_vel[i, 12:14] = [V_l, V_r]

    # Position PID -> desired velocity, capped at v_max.
    dy_des = float(np.clip(y_pos_ctrl(state.y, state.dy, y0),
                           -v_max, v_max))
    dz_des = float(np.clip(z_pos_ctrl(state.z, state.dz, z0),
                           -v_max, v_max))
    # Velocity PID -> desired tilt, capped at tilt_max.
    phi_des = float(np.clip(y_vel_ctrl(state.dy, 0, dy_des),
                             -tilt_max, tilt_max))
    z_corr   = z_vel_ctrl(state.dz, 0, dz_des)
    phi_fb   = phi_ctrl(state.phi, state.dphi, phi_des)
    hover_ff = V_hover / np.cos(state.phi)
    V_l, V_r = zphi2V(hover_ff + z_corr, phi_fb)
    set_v_nohover(drone, V_l, V_r)

In [ ]:
# Compare the two controllers on the same step: position, velocity, tilt.
t = np.arange(N) / 60.0
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].axhline(y0, color='k', lw=0.5, ls='--', alpha=0.5, label='setpoint')
axes[0].plot(t, results_flat[:, 1], color='C3', label='flat cascade (no velocity loop)')
axes[0].plot(t, results_vel[:,  1], color='C0', label='velocity-loop cascade')
axes[0].set_ylabel('y [m]')
axes[0].legend(loc='lower right')
axes[0].grid()

axes[1].axhline( v_max, color='k', lw=0.5, ls='--', alpha=0.5)
axes[1].axhline(-v_max, color='k', lw=0.5, ls='--', alpha=0.5, label='+/- v_max')
axes[1].plot(t, results_flat[:, 4], color='C3')
axes[1].plot(t, results_vel[:,  4], color='C0')
axes[1].set_ylabel('dy [m/s]')
axes[1].legend(loc='upper right')
axes[1].grid()

axes[2].axhline( np.rad2deg(tilt_max), color='k', lw=0.5, ls='--', alpha=0.5)
axes[2].axhline(-np.rad2deg(tilt_max), color='k', lw=0.5, ls='--', alpha=0.5, label='+/- 20 deg tilt clip')
axes[2].plot(t, np.rad2deg(results_flat[:, 3]), color='C3')
axes[2].plot(t, np.rad2deg(results_vel[:,  3]), color='C0')
axes[2].set_ylabel('phi [deg]')
axes[2].set_xlabel('t [s]')
axes[2].legend(loc='upper right')
axes[2].grid()

plt.tight_layout()

### Task: who sets the speed?

In the cell with the controllers, set `v_max = 0.5` and run the step again: the drone now cruises at 0.5 m/s. Put `v_max` back to 1.0 and change the gain of the position loop to `Kp = -4.0` instead of `-1.6`: the cruise speed stays at 1 m/s, only the braking at the end changes (the drone overshoots a little and is still settling when the run ends). In the velocity cascade the speed is set by the clip, not by the gain. Now double the gain of the flat cascade instead (`y_PID_flat`, `Kp = 3.0`) and look at what happens to its speed.

## Comparison

The flat cascade hits the tilt clip, accelerates the drone to several m/s, and overshoots the setpoint by 1 m or more before braking. The velocity cascade respects $v_{\mathrm{max}} = 1$ m/s as a hard upper bound on the drone's *speed*, cruises through the middle of the move at that speed, and settles at the setpoint without overshoot.

Two reasons to prefer the velocity loop as a building block:

1. **The limit acts on a physical quantity.** $v_{\mathrm{max}} = 1$ m/s is a physical bound on the drone's speed. The $\pm 20°$ tilt clip in the flat cascade was a bound on a dimensionless feedback signal whose relationship to speed depends on the gain.

2. **The velocity loop damps the motion.** When the drone is moving faster than its velocity setpoint, the velocity error directly produces a tilt with the right sign and magnitude to brake. The flat cascade had to wait for a position error to swing the other way before it could do anything about excess speed.

In the later notebooks the drag of the simulator (`drag_t=4`) is on by default. The drag limits the speed of the drone, which is why the flat cascade worked well enough there: the drone never became fast enough for the missing speed limit to matter. Without drag, as above, the same cascade overshoots by metres. A controller should not rely on the drag of the platform for its speed limit, but contain the limit itself.

This is the architecture the real Crazyflie firmware uses (Section 3 of the paper cited in `drone_parameters.md`), minus a fourth attitude-rate loop that runs at 500 Hz inside the microcontroller and that we deliberately do not model here.

From the next notebook onwards every notebook runs the same five-target race, so the results compare directly: first the velocity-loop cascade through the track, then, among the extra notebooks, the same race flown by hand with the keyboard moving the setpoint, a controller that inverts the equations of motion of the drone, and smooth trajectories through the waypoints on top of it.